## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

color_palette = sns.color_palette()

## Read data

In [ ]:
import pandas as pd
from pathlib import Path


data_file_name = Path('...')
data = pd.read_csv(data_file_name, parse_dates=['...'], index_col='...')

## Sanity check

In [ ]:
def sanity_check(df: pd.DataFrame):
    """Perform sanity checks on time series data."""
    # check data info
    df.info()
    df.head()
    df.describe()

    # check if index is sorted
    print(f'\nData index is sorted: {df.index.is_monotonic_increasing}')

    # check for duplicate indices
    print(f'\nDuplicate indices present: {df.index.duplicated().any()}')

    # check time differences to see if the time series is regular
    print(f'\nTime differences:\n{df.index.to_series().diff().value_counts().head()}')

    # check for missing values
    print(f'\nMissing values proportion:\n{df.isna().mean()}')

    # check for outliers using boxplot
    plt.boxplot(df.select_dtypes(include=[np.number]).dropna().values)
    plt.show()


sanity_check(data)

## Data exploration (cross-feature effects)

#### Plot series

In [ ]:
data['...'].plot(style='.', color='...')

data['...'].plot(kind='hist', bins=500)
data.query('column < 100').plot(style='.')

data['...'].groupby('grouping_col')['values_col'].mean()

#### Distributions, correlations

In [ ]:
pd.plotting.scatter_matrix(data)
data.corr()
data['column'].plot(kind='hist', bins=500)

## Temporal structure

#### Autocovariance

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(y, lags=50)
plot_pacf(y, lags=50)
plt.tight_layout()
plt.show()

#### Stationarity

In [ ]:
from statsmodels.tsa.stattools import adfuller


def check_stationarity(time_series: pd.Series, significance_level: float = 0.05) -> None:
    """Check stationarity of a time series using the Augmented Dickey-Fuller test."""
    adfuller_result = adfuller(time_series)
    p_value = adfuller_result[1]
    if p_value < significance_level:
        print(f'The time series is stationary (p-value: {p_value:.4f}).')
    else:
        print(f'The time series is non-stationary (p-value: {p_value:.4f}).')

#### Box-Cox transform of heteroskedastic features

In [ ]:
from scipy.stats import boxcox, inv_boxcox  # Box-Cox transformation

y_train_box_cox_array, lambda_boxcox = boxcox(y_train + 1)  # shift to avoid zero values
y_train_box_cox = pd.Series(y_train_box_cox_array, index=y_train.index)

y_test_box_cox_array, _ = boxcox(y_test + 1, lmbda=lambda_boxcox)  # use the same lambda for test set
y_test_box_cox = pd.Series(y_test_box_cox_array, index=y_test.index)

y_train_box_cox.plot(title='Box-Cox Transformed Time Series')

########################################
# at prediction:
box_cox_forecast = model.predict(X_test)
y_hat = pd.Series(inv_boxcox(box_cox_forecast, lambda_boxcox), index=y_test.index)

#### Seasonality

Evenly spaced data

In [ ]:
# requirements: time series is stationary and has a regular time step

from numpy.fft import rfft, rfftfreq
from scipy.signal import detrend


def check_seasonality(time_series: pd.Series, power_threshold: float) -> None:
    # to apply Fourier, detrend the data first
    time_series_centered = time_series - np.mean(time_series)
    time_series_detrended = detrend(time_series_centered)

    fft_values = rfft(time_series_detrended)
    time_step = 1.
    frequencies = rfftfreq(len(time_series_detrended), d=time_step)
    power = np.abs(fft_values)**2  # power = amplitude squared

    plt.plot(1. / frequencies[1:], power[1:])  # plot period vs power
    plt.xlabel("Period")
    plt.ylabel("Power")
    # plt.xlim(0, 250)
    plt.title("Power Spectrum")
    plt.show()

    print(f'Peaks at periods: {(power > power_threshold).nonzero()[0] + 1}')

Unevenly spaced data

In [ ]:
from astropy.timeseries import LombScargle

t = y_train.index
x = y_train.values

# t = (t - t[0]) / np.timedelta64(1, 'W')  # example: weeks
# number of months

x = x - x.mean()

freqs, power = LombScargle(range(len(x)), x).autopower()

plt.plot(1. / freqs, power)
plt.xlim(0, 55)
plt.xlabel("Period (hours)")
plt.ylabel("Power")
plt.show()

## Train-test split

#### Simple

In [ ]:
X = data.drop(columns=['target'])
y = data['target']

# create a test set with the last 20% of the data
test_data_ratio = 0.2
split_idx = int(len(data) * (1 - test_data_ratio))

all_data_train = data.iloc[:split_idx]
all_data_test = data.iloc[split_idx:]
X_train = X.iloc[:split_idx]
y_train = y.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_test = y.iloc[split_idx:]

fig, ax = plt.subplots(figsize=(..., ...))
y_train.plot(ax=ax, label='Train')
y_test.plot(ax=ax, label='Test')
plt.show()

#### Cross validation

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error


FEATURES = ['...', ...]
TARGET = '...'

tss = TimeSeriesSplit(n_splits=5, test_size=..., gap=...)


preds = []
scores = []
for fold, (train_idx, val_idx) in enumerate(tss.split(data)):
    train = data.iloc[train_idx]
    test = data.iloc[val_idx]

    train = create_features(train)
    test = create_features(test)

    X_train = train[FEATURES]
    y_train = train[TARGET]
    X_test = test[FEATURES]
    y_test = test[TARGET]

    reg.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=100)
    y_pred = reg.predict(X_test)
    preds.append(y_pred)
    scores.append(root_mean_squared_error(y_test, y_pred))

## Features

### New features

#### Calendar features

In [ ]:
def create_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.day_of_week
    df['week'] = df.index.week
    return df

#### Lag features

In [ ]:
# use lags for seasonality lengths

LAGS = [pd.Timedelta('364 days'), ...]

def create_lagged_features(
    data: pd.DataFrame,
    feature_cols: list[str],
    lags: list[pd.Timedelta],
) -> pd.DataFrame:
    """Create lagged features."""
    for col in feature_cols:
        target_map = data[col].to_dict()
        for lag in lags:
            data[f'{col}_lag_{lag}'] = (data.index - lag).map(target_map)
    return data

#### Seasonality features

In [ ]:

def create_fourier_features(df: pd.DataFrame, periods: list[int], feature_cols: list[str]) -> pd.DataFrame:
    """Create Fourier features for a given list of periods."""
    for period in periods:
        df[f'fourier_sin_{period}'] = np.sin(2 * np.pi * df['month_num'] / period)
        df[f'fourier_cos_{period}'] = np.cos(2 * np.pi * df['month_num'] / period)
        for col in feature_cols:
            # create the value from the previous periods
            df[f'{col}_lag_{period}'] = df[col].shift(period)
            df[f'{col}_lag_{2 * period}'] = df[col].shift(2 * period)
            df[f'{col}_lag_{3 * period}'] = df[col].shift(3 * period)

    return df

#### Rolling features

In [ ]:
def add_rolling(data: pd.DataFreame, feature_cols: list[str], windows: list[int]) -> pd.DataFrame:
    for col in feature_cols:
        for w in windows:
            data[f"{col}_roll_mean_{w}"] = data[col].shift(1).rolling(w).mean()
            data[f"{col}_roll_std_{w}"] = data[col].shift(1).rolling(w).std()
    return data

#### Decomposition

In [ ]:
from statsmodels.tsa.seasonal import MSTL


# decomposition
def decompose(series: pd.Series) -> tuple[pd.Series, pd.DataFrame, pd.Series]:
    mstl = MSTL(series, periods=(24, 168), lmbda="auto", robust=True)
    res = mstl.fit()

    return res.trend, res.seasonal, res.resid


# forecast trend
import numpy as np
from sklearn.linear_model import LinearRegression
t = np.arange(len(trend)).reshape(-1, 1)
model = LinearRegression()
model.fit(t, trend)
horizon = 12
t_future = np.arange(len(trend), len(trend) + horizon).reshape(-1, 1)
trend_forecast = model.predict(t_future)


# forecast seasonal parts (repeat last cycles)
...


# combine forecasts
forecast = trend_forecast + seasonal_forecast + resid_forecast

### Feature selection

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5)
lasso.fit(X, y)

selected_features = X.columns[lasso.coef_ != 0]

## Modeling

#### Linear regression

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge()
ridge.fit(X_train_nona, y_train_nona, alpha=1.0)
y_pred_test = ridge.predict(X_test_nona)

#### XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    base_score=0.,
    booster='gbtree',
    n_estimators=500,
    max_depth=3,
    learning_rate=1e-1,
    early_stopping_rounds=10,
    eval_metric='rmse',
    random_state=42
)

xgb.fit(X_train_nona, y_train_nona, eval_set=[(X_test_nona, y_test_nona)], verbose=100)

y_pred_test = pd.Series(
    xgb.predict(X_test_nona),
    index=y_test_nona.index
)

#### Random forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_test = rf.predict(X_test)

#### Neural network

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset


class Net(nn.Module):

    def __init__(self, input_size, hidden_size: list[int]):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size[0]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size[0], hidden_size[1]),
            nn.ReLU(),
            nn.Linear(hidden_size[1], 1)
        )

    def forward(self, x):
        return self.net(x)


input_size = X_train.shape[1]
hidden_size = 64
net = Net(input_size, hidden_size)

loss_function = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=False)  # no shuffle for time series

num_epochs = 100
best_loss = float('inf')
patience = 10  # number of epochs to wait for improvement before stopping
counter = 0
for epoch in range(num_epochs):
    net.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        preds = net(xb)
        loss = loss_function(preds, yb)
        loss.backward()
        optimizer.step()

    net.eval()
    with torch.no_grad():
        val_preds = net(X_test_tensor)
        val_loss = loss_function(val_preds, y_test_tensor)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Train: {loss.item():.4f}, Val: {val_loss.item():.4f}')

    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

## Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [1e-2, 1e-1, 1e0, 1e1, 1e2]}
grid_search = GridSearchCV(model, param_grid, cv=5)
grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

## Feature importance

In [ ]:
feature_importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feature_importances.plot(kind='barh', title='Feature Importances')

## Error analysis

In [ ]:
data['error'] = np.abs(y_test - y_pred)
data.groupby('hour')['error'].mean().sort_values()